# Assignment 2.2: Naive Bayes on tabular data

The Naive Bayes variants are already implemented in this notebook. Your job is to decide which
variant belongs on which features, call it with the right arguments, run the experiments, and
write the report.

Four code cells are yours: `run_digits`, the two Titanic column lists, `run_titanic`, and the
scikit-learn run table. Everything else is provided and already runs.

# Import libraries
Do not use any other Python library.

numpy - Linear algebra library for handling vectors and matrices, collectively processed as numpy arrays.

matplotlib - Graphing library for visualizing results.

csv - Library used for reading csv files into python lists.

sklearn - Machine learning library from which we source one dataset, the confusion matrix plot, and the reference Naive Bayes implementations you compare your own results against.

time - Simple library for timing code.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from csv import reader
from sklearn.datasets import load_digits
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from time import time

# Load Datasets

These functions load the two tabular datasets.

`load_digits_data` returns the 1797 handwritten digits, each one 64 pixels with an intensity from
0 to 16.

Outputs:

*   **features**: list of N rows, each one a list of 64 pixel values
*   **labels**: list of N integer class labels
*   **label_names**: list of class names, indexed by the label

`load_titanic` returns the 891 passengers as columns, so you can pick out the columns you want.
PassengerId, Name and Ticket are dropped because they identify a passenger rather than describe
one, and Cabin is dropped because it is missing for most passengers. A value that is blank in the
file becomes `None`, which the training functions leave out of their counts and their means.

Outputs:

*   **column_values**: dictionary mapping each column name to its list of N values, as strings or None
*   **labels**: list of N integer class labels
*   **label_names**: list of class names, indexed by the label

In [ ]:
def load_digits_data(plot=True):
    # optical digits dataset from scikit-learn
    digits = load_digits()
    features = digits.data.tolist()
    labels = [int(label) for label in digits.target]

    if plot:
        images = digits.data.reshape(-1, 8, 8)
        plt.figure(figsize=(10, 5))
        for i in range(10):
            plt.subplot(3, 4, i + 1)
            plt.imshow(images[i], cmap='gray')
            plt.title(f'Handwritten {labels[i]}')
            plt.axis('off')
        plt.tight_layout()
        plt.show()

    return features, labels, [str(digit) for digit in range(10)]


def load_titanic(plot=True):
    # titanic dataset from https://www.kaggle.com/datasets/yasserh/titanic-dataset
    with open('../data/classification-datasets/Titanic-Dataset.csv', 'r') as file:
        rows = list(reader(file))
    header = rows[0]
    # identifiers, free text, and one column that is missing for most passengers
    dropped = ['PassengerId', 'Survived', 'Name', 'Ticket', 'Cabin']
    kept = [name for name in header if name not in dropped]
    positions = {name: header.index(name) for name in kept}

    column_values = {name: [] for name in kept}
    labels = []
    for row in rows[1:]:
        for name in kept:
            value = row[positions[name]]
            # a blank cell is a missing value, not a category of its own
            column_values[name].append(None if value == '' else value)
        labels.append(int(row[header.index('Survived')]))

    if plot:
        plt.figure(figsize=(10, 4))
        for position, name in enumerate(['Sex', 'Pclass']):
            values = sorted({value for value in column_values[name] if value is not None})
            survived = [sum(1 for value, label in zip(column_values[name], labels)
                            if value == group and label == 1) for group in values]
            died = [sum(1 for value, label in zip(column_values[name], labels)
                        if value == group and label == 0) for group in values]
            plt.subplot(1, 2, position + 1)
            plt.bar(values, died, label='died')
            plt.bar(values, survived, bottom=died, label='survived')
            plt.xlabel(name)
            plt.ylabel('passengers')
            plt.legend()
        plt.tight_layout()
        plt.show()

    return column_values, labels, ['died', 'survived']

# Function: train_prior

Estimates p(y = l) as the fraction of the training labels that carry class l. That is the prior
term in Equations 1 and 2 of the spec.

Inputs:
*   **labels**: list of training labels

Output:
*   **class_priors**: dictionary mapping each class to its prior probability (float)

In [ ]:
def train_prior(labels):
    class_counts = {}
    for label in labels:
        class_counts[label] = class_counts.get(label, 0) + 1
    return {label: count / len(labels) for label, count in class_counts.items()}

# Function: train_gaussian

Fits one Gaussian per class per column, the parameters Equation 6 needs. The mean and the variance
are the maximum likelihood estimates over the training rows of that class, and missing values are
left out of both. `variance_epsilon` is then added to every variance: on the digits data 20 of the
64 pixels are constant within at least one class, and a zero variance makes Equation 6 undefined.

Inputs:
*   **rows**: list of rows, each one a list of numbers or None
*   **labels**: list of labels, one per row
*   **variance_epsilon**: constant added to every variance (float)

Output:
*   **model**: dictionary holding the per class means and variances

# Function: predict_gaussian

Scores every row under every class with Equation 2, using the Gaussian density p_G(x_i | y = l) of
Equation 6 for each column. A missing value contributes no term. Pass `class_priors` to include the prior term,
or leave it out to get the likelihood alone.

Inputs:
*   **rows**: list of rows, each one a list of numbers or None
*   **model**: the dictionary returned by train_gaussian
*   **class_priors**: dictionary mapping each class to its prior probability, or None for no prior term

Output:
*   **log_probabilities**: list with one dictionary per row, mapping each class to its log probability

In [ ]:
def train_gaussian(rows, labels, variance_epsilon=0.01):
    n_features = len(rows[0])
    class_values = {}
    for row, label in zip(rows, labels):
        columns = class_values.setdefault(label, [[] for _ in range(n_features)])
        for i, value in enumerate(row):
            if value is not None:
                columns[i].append(value)

    means = {}
    variances = {}
    for label, columns in class_values.items():
        means[label] = [float(np.mean(column)) if column else 0.0 for column in columns]
        variances[label] = [float(np.var(column)) + variance_epsilon if column
                            else variance_epsilon for column in columns]
    return {'means': means, 'variances': variances}


def predict_gaussian(rows, model, class_priors=None):
    # the constant part of the density does not depend on the row
    log_normalizers = {label: [-0.5 * np.log(2 * np.pi * variance) for variance in variances]
                       for label, variances in model['variances'].items()}

    log_probabilities = []
    for row in rows:
        class_log_probabilities = {}
        for label, means in model['means'].items():
            variances = model['variances'][label]
            normalizers = log_normalizers[label]
            total = np.log(class_priors[label]) if class_priors else 0.0
            for i, value in enumerate(row):
                if value is None:
                    continue
                difference = value - means[i]
                total += normalizers[i] - difference * difference / (2 * variances[i])
            class_log_probabilities[label] = total
        log_probabilities.append(class_log_probabilities)
    return log_probabilities

# Function: train_categorical

Collects the counts Equation 5 needs: for each class and each column, how many rows take each
value and how many rows are not missing that column. The set of values a column can take is
whatever the training rows contain, which is the size that scales the smoothing in the
denominator. Here k is the Laplace smoothing constant, which adds k to every count so that no
probability is ever zero. This project uses k = 1.

Inputs:
*   **rows**: list of rows, each one a list of values or None
*   **labels**: list of labels, one per row
*   **smoothing**: Laplace smoothing constant k (float)

Output:
*   **model**: dictionary holding the per class value counts, the per class counts of rows that are not missing, the values seen in each column, and the smoothing constant

# Function: predict_categorical

Scores every row under every class with Equation 2, using the Categorical estimate
p_C(x_i = c | y = l) of Equation 5 for each column. A missing value contributes no term, and a value that never appeared in training
falls back to the smoothed numerator alone. Pass `class_priors` to include the prior term, or
leave it out to get the likelihood alone.

Inputs:
*   **rows**: list of rows, each one a list of values or None
*   **model**: the dictionary returned by train_categorical
*   **class_priors**: dictionary mapping each class to its prior probability, or None for no prior term

Output:
*   **log_probabilities**: list with one dictionary per row, mapping each class to its log probability

In [ ]:
def train_categorical(rows, labels, smoothing=1):
    n_features = len(rows[0])
    value_counts = {}
    present_counts = {}
    feature_values = [set() for _ in range(n_features)]
    for row, label in zip(rows, labels):
        counts = value_counts.setdefault(label, [{} for _ in range(n_features)])
        present = present_counts.setdefault(label, [0] * n_features)
        for i, value in enumerate(row):
            # a missing value is left out of the counts
            if value is None:
                continue
            counts[i][value] = counts[i].get(value, 0) + 1
            present[i] += 1
            feature_values[i].add(value)
    return {'value_counts': value_counts, 'present_counts': present_counts,
            'feature_values': feature_values, 'smoothing': smoothing}


def predict_categorical(rows, model, class_priors=None):
    smoothing = model['smoothing']
    n_values = [len(values) for values in model['feature_values']]

    # one log probability table per class and column, built once
    log_probability_tables = {}
    unseen_log_probabilities = {}
    for label, counts in model['value_counts'].items():
        present = model['present_counts'][label]
        tables = []
        unseen = []
        for i, column_counts in enumerate(counts):
            denominator = present[i] + smoothing * n_values[i]
            tables.append({value: np.log((count + smoothing) / denominator)
                           for value, count in column_counts.items()})
            unseen.append(np.log(smoothing / denominator))
        log_probability_tables[label] = tables
        unseen_log_probabilities[label] = unseen

    log_probabilities = []
    for row in rows:
        class_log_probabilities = {}
        for label, tables in log_probability_tables.items():
            unseen = unseen_log_probabilities[label]
            total = np.log(class_priors[label]) if class_priors else 0.0
            for i, value in enumerate(row):
                if value is None:
                    continue
                total += tables[i].get(value, unseen[i])
            class_log_probabilities[label] = total
        log_probabilities.append(class_log_probabilities)
    return log_probabilities

# Function: categorical_table

Pulls the named columns out of `column_values` and returns them row by row, values untouched.

# Function: numeric_table

Pulls the named columns out of `column_values` and returns them row by row as floats. A missing
value stays None. A column that does not hold numbers raises a ValueError.

Inputs, both:
*   **column_values**: dictionary mapping each column name to its list of values
*   **names**: list of the column names to pull, in the order you want them

Output, both:
*   **rows**: list of N rows, each one a list with one entry per name

# Function: combine_log_probabilities

Adds two sets of per class log probabilities row by row, the p_C sum and the p_G sum of
Equation 7.

Inputs:
*   **first**: list with one dictionary per row, mapping each class to its log probability
*   **second**: a second list of the same shape and row order

Output:
*   **log_probabilities**: list with one dictionary per row, holding the sums

In [ ]:
def categorical_table(column_values, names):
    n_rows = len(next(iter(column_values.values())))
    return [[column_values[name][i] for name in names] for i in range(n_rows)]


def numeric_table(column_values, names):
    n_rows = len(next(iter(column_values.values())))
    return [[None if column_values[name][i] is None else float(column_values[name][i])
             for name in names] for i in range(n_rows)]


def combine_log_probabilities(first, second):
    return [{label: row[label] + other[label] for label in row}
            for row, other in zip(first, second)]

# Function: predicted_labels

Turns per class log probabilities into one label per instance, the arg max in Equations 2
and 7.

Input:
*   **log_probabilities**: list with one dictionary per instance, mapping each class to its log probability

Output:
*   **predicted**: list of predicted labels, one per instance

# Function: accuracy

Percentage of instances whose predicted label matches the true label.

Inputs:
*   **true_labels**: list of true labels
*   **predicted**: list of predicted labels

Output:
*   **accuracy**: percentage of correct predictions (float)

# Function: show_confusion_matrix

Draws one confusion matrix. Rows are true classes, columns are predicted classes.

Inputs:
*   **true_labels**: list of true labels
*   **predicted**: list of predicted labels
*   **label_names**: list of class names, indexed by the label
*   **title**: title for the figure (string)

In [ ]:
def predicted_labels(log_probabilities):
    return [max(row, key=row.get) for row in log_probabilities]


def accuracy(true_labels, predicted):
    correct = sum(1 for true, prediction in zip(true_labels, predicted) if true == prediction)
    return correct / len(true_labels) * 100


def show_confusion_matrix(true_labels, predicted, label_names, title):
    matrix = confusion_matrix(true_labels, predicted)
    display = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=label_names)
    # ten digit classes need a bigger figure than two titanic classes
    size = 5 if len(label_names) <= 4 else 8
    figure, axes = plt.subplots(figsize=(size, size))
    display.plot(cmap=plt.cm.Blues, ax=axes, colorbar=False, values_format='d')
    axes.set_title(title)
    plt.show()

# Function: fold_indices

Shuffles the instance indices once with seed 42 and cuts them into k folds. Each fold is the test
set exactly once, so every instance is tested exactly once across the k runs.

Inputs:
*   **n_samples**: number of instances (integer)
*   **k_folds**: number of folds (integer)
*   **seed**: seed for the shuffle (integer)

Output:
*   **splits**: list of (train_indices, test_indices) pairs, one per fold

# Function: cross_validate

Runs one experiment function over the k folds and collects the results. The experiment function is
called once per fold as `run_experiment(*train_tables, train_labels, *test_tables)` and returns one
predicted label per test instance. Titanic passes two tables, the categorical one and the numeric
one, so its experiment function takes five arguments.

Inputs:
*   **tables**: list of feature tables, each one N rows long and in the same row order
*   **labels**: list of N labels
*   **run_experiment**: function that trains on a fold and returns predicted labels
*   **k_folds**: number of folds (integer)

Outputs:
*   **fold_accuracies**: list of the accuracy on each held out fold (floats)
*   **true_labels**: true labels of every instance, in the order they were tested
*   **predicted**: predicted labels of every instance, in the same order

In [ ]:
def fold_indices(n_samples, k_folds=5, seed=42):
    order = np.random.default_rng(seed).permutation(n_samples)
    folds = np.array_split(order, k_folds)
    splits = []
    for held_out in range(k_folds):
        train_indices = [int(i) for fold in range(k_folds) if fold != held_out
                         for i in folds[fold]]
        test_indices = [int(i) for i in folds[held_out]]
        splits.append((train_indices, test_indices))
    return splits


def cross_validate(tables, labels, run_experiment, k_folds=5):
    fold_accuracies = []
    true_labels = []
    predicted = []
    for train_indices, test_indices in fold_indices(len(labels), k_folds):
        train_tables = [[table[i] for i in train_indices] for table in tables]
        test_tables = [[table[i] for i in test_indices] for table in tables]
        train_labels = [labels[i] for i in train_indices]
        fold_true = [labels[i] for i in test_indices]
        fold_predicted = run_experiment(*train_tables, train_labels, *test_tables)
        fold_accuracies.append(accuracy(fold_true, fold_predicted))
        true_labels.extend(fold_true)
        predicted.extend(fold_predicted)
    return fold_accuracies, true_labels, predicted

# Function: run_library_numeric

Wraps a scikit-learn classifier so the cross validation harness can run it. Returns a run function
with the same inputs and output as `run_digits`: the rows become a numpy float array, the
classifier is fitted on the training fold, and the held out rows are predicted.

The library is not doing quite what the provided code does. `GaussianNB` smooths its variances with
`var_smoothing`, a fraction of the largest variance over all the features, where the provided code
adds the same constant 0.01 to every variance. The two models can therefore disagree on a row.

Inputs:
*   **model**: an unfitted scikit-learn classifier

Output:
*   **run_experiment**: function that runs one fold, with the same signature as `run_digits`

# Function: run_library_titanic

Builds the scikit-learn version of the Titanic model, `CategoricalNB` on the categorical columns and
`GaussianNB` on the numeric ones, added together as in Equation 7. Returns a run function with the
same inputs and output as `run_titanic`.

Two things differ from the provided code. scikit-learn has no missing value, so a blank numeric cell
is filled with the mean of that column over the training fold instead of contributing no term.
`CategoricalNB` takes integer codes, so each categorical column is numbered from a code table built
on the training rows alone, with one extra code kept for a value that is missing or that never
appeared in training. The two models are combined through `predict_log_proba`, which already carries
the class prior, so one copy of the prior is subtracted back out and p(y = l) is counted once.

Inputs:
*   **categorical_columns**: list of the column names the categorical variant handles
*   **numeric_columns**: list of the column names the numeric variant handles
*   **smoothing**: Laplace smoothing constant k, which scikit-learn calls alpha (float)

Output:
*   **run_experiment**: function that runs one fold, with the same signature as `run_titanic`

In [ ]:
def run_library_numeric(model):
    def run_experiment(train_features, train_labels, test_features):
        train_rows = np.array(train_features, dtype=float)
        test_rows = np.array(test_features, dtype=float)
        model.fit(train_rows, train_labels)
        return [int(label) for label in model.predict(test_rows)]
    return run_experiment


def run_library_titanic(categorical_columns, numeric_columns, smoothing=1):
    n_categorical = len(categorical_columns)
    n_numeric = len(numeric_columns)

    def code_tables_from(rows):
        # one code table per column, numbered from the training values alone
        tables = []
        for i in range(n_categorical):
            values = sorted({row[i] for row in rows if row[i] is not None})
            tables.append({value: code for code, value in enumerate(values)})
        return tables

    def encode(rows, tables):
        # the code past the end of a table is the one kept for missing or unseen
        return np.array([[tables[i].get(row[i], len(tables[i])) for i in range(n_categorical)]
                         for row in rows], dtype=int)

    def fill(rows, column_means):
        # sklearn has no missing value, so a blank cell takes the training fold mean
        return np.array([[column_means[i] if row[i] is None else float(row[i])
                          for i in range(n_numeric)] for row in rows], dtype=float)

    def run_experiment(train_category_rows, train_number_rows, train_labels,
                       test_category_rows, test_number_rows):
        tables = code_tables_from(train_category_rows)
        min_categories = [len(table) + 1 for table in tables]
        column_means = [float(np.mean([row[i] for row in train_number_rows if row[i] is not None]))
                        for i in range(n_numeric)]

        categorical_model = CategoricalNB(alpha=smoothing, min_categories=min_categories)
        categorical_model.fit(encode(train_category_rows, tables), train_labels)
        gaussian_model = GaussianNB()
        gaussian_model.fit(fill(train_number_rows, column_means), train_labels)

        categorical_log = categorical_model.predict_log_proba(encode(test_category_rows, tables))
        gaussian_log = gaussian_model.predict_log_proba(fill(test_number_rows, column_means))
        # both terms carry the prior, so one copy comes back out and p(y = l) is counted once
        log_probabilities = categorical_log + gaussian_log - categorical_model.class_log_prior_
        return [int(categorical_model.classes_[index])
                for index in np.argmax(log_probabilities, axis=1)]

    return run_experiment


# the driver cells below fill these in as they run
provided_accuracy = {}
library_accuracy = {}

# Function: run_digits

Runs the handwritten digits experiment on one fold. The cross validation harness calls this once
per fold with that fold's training rows and labels and the held out rows, and expects one
predicted label per held out row. Treat the pixel values as numeric and use
`variance_epsilon=0.01`.

Inputs:
*   **train_features**: list of training rows, each one a list of 64 pixel values
*   **train_labels**: list of training labels
*   **test_features**: list of held out rows

Output:
*   **predicted**: list of predicted labels, one per held out row

In [ ]:
def run_digits(train_features, train_labels, test_features):
    # 1. estimate the class priors from the training labels
    # 2. train the variant that fits a bell curve to each column, with variance epsilon 0.01
    # 3. score the held out rows with the predict function that matches that variant, priors included
    # 4. return one label per held out row
    # TODO
    pass

# Handwritten digits

Runs 5 fold cross validation on the digits, prints the accuracy on each fold and the mean over the
folds, and draws one confusion matrix covering all five test folds.

In [ ]:
features, labels, label_names = load_digits_data(plot=True)
print(f"digits: {len(features)} rows, {len(features[0])} features, {len(label_names)} classes")

start_time = time()
fold_accuracies, true_labels, predicted = cross_validate([features], labels, run_digits, k_folds=5)
elapsed_time = time() - start_time

print("fold accuracies: " + ", ".join(f"{value:.2f}%" for value in fold_accuracies))
print(f"mean accuracy: {np.mean(fold_accuracies):.2f}%")
provided_accuracy["digits"] = float(np.mean(fold_accuracies))
print(f"elapsed time: {elapsed_time:.1f}s")
show_confusion_matrix(true_labels, predicted, label_names, "digits, gaussian")

# Titanic

The Titanic rows mix two kinds of feature, so the model is two variants added together. Load the
data first and look at the columns that are left after the identifiers are dropped.

In [ ]:
column_values, titanic_labels, titanic_label_names = load_titanic(plot=True)
print(f"titanic: {len(titanic_labels)} rows, {len(titanic_label_names)} classes")
print("available columns:", list(column_values))
for name, values in column_values.items():
    present = [value for value in values if value is not None]
    print(f"  {name:10s} {len(set(present)):3d} distinct values, {len(values) - len(present):3d} missing")

# Titanic columns

Split the available columns between the two lists. Every column goes in exactly one list, and the
spec says which variant handles which kind of feature.

In [ ]:
# 1. list the columns you will hand to the categorical variant
# 2. list the columns you will hand to the numeric variant
# TODO
categorical_columns, numeric_columns = [], []

# Function: run_titanic

Runs the Titanic experiment on one fold. The cross validation harness calls this once per fold
with the training half of both tables, the training labels, and the held out half of both tables,
and expects one predicted label per held out row. Equation 7 combines the two variants. The prior
p(y = l) appears once, so it goes into exactly one of the two prediction calls, and a feature whose
value is missing from an instance contributes no term. Use smoothing k = 1 and
`variance_epsilon=0.01`.

Inputs:
*   **train_category_rows**: training rows of the categorical columns
*   **train_number_rows**: training rows of the numeric columns
*   **train_labels**: list of training labels
*   **test_category_rows**: held out rows of the categorical columns
*   **test_number_rows**: held out rows of the numeric columns

Output:
*   **predicted**: list of predicted labels, one per held out row

In [ ]:
def run_titanic(train_category_rows, train_number_rows, train_labels,
                test_category_rows, test_number_rows):
    # 1. estimate the class priors from the training labels
    # 2. train one variant on the categorical rows and the other on the numeric rows
    # 3. score the held out rows with each variant, giving the priors to one call and not the other
    # 4. add the two sets of per class log probabilities together
    # 5. return one label per held out row
    # TODO
    pass

# Main Titanic code

Builds the two tables from your column lists, runs 5 fold cross validation, prints the accuracy on
each fold and the mean over the folds, and draws one confusion matrix covering all five test
folds.

In [ ]:
category_rows = categorical_table(column_values, categorical_columns)
number_rows = numeric_table(column_values, numeric_columns)
print(f"categorical columns: {categorical_columns}")
print(f"numeric columns: {numeric_columns}")

start_time = time()
fold_accuracies, true_labels, predicted = cross_validate(
    [category_rows, number_rows], titanic_labels, run_titanic, k_folds=5)
elapsed_time = time() - start_time

print("fold accuracies: " + ", ".join(f"{value:.2f}%" for value in fold_accuracies))
print(f"mean accuracy: {np.mean(fold_accuracies):.2f}%")
provided_accuracy["titanic"] = float(np.mean(fold_accuracies))
print(f"elapsed time: {elapsed_time:.1f}s")
show_confusion_matrix(true_labels, predicted, titanic_label_names, "titanic, categorical and gaussian")

# scikit-learn runs

The provided code is the reference for what each variant computes. Now run the matching
scikit-learn classes over the same folds. `run_library_numeric` takes one unfitted classifier.
`run_library_titanic` takes your two column lists and the smoothing constant k, which scikit-learn
calls alpha. Give each dataset the class that matches the variant you used above.

In [ ]:
# 1. wrap the library class that matches the variant you ran on the digits
# 2. build the titanic run function from your two column lists and the smoothing constant
# TODO
library_runs = {}

# Main scikit-learn code

Runs every entry of your table over the same 5 folds as the provided code, prints the accuracy on
each fold and the mean over the folds, and then prints the two mean accuracies side by side with
the difference between them.

In [ ]:
library_tables = {
    "digits": ([features], labels),
    "titanic": ([category_rows, number_rows], titanic_labels),
}

for dataset_name, run_experiment in library_runs.items():
    tables, dataset_labels = library_tables[dataset_name]
    print(f"RUNNING {dataset_name}, scikit-learn")
    start_time = time()
    fold_accuracies, true_labels, predicted = cross_validate(
        tables, dataset_labels, run_experiment, k_folds=5)
    elapsed_time = time() - start_time

    library_accuracy[dataset_name] = float(np.mean(fold_accuracies))
    print("fold accuracies: " + ", ".join(f"{value:.2f}%" for value in fold_accuracies))
    print(f"mean accuracy: {np.mean(fold_accuracies):.2f}%")
    print(f"elapsed time: {elapsed_time:.1f}s")

print("mean accuracy over 5 folds, provided code against scikit-learn")
for dataset_name in ["digits", "titanic"]:
    provided = provided_accuracy.get(dataset_name)
    library = library_accuracy.get(dataset_name)
    if provided is None or library is None:
        print(f"  {dataset_name:8s} not run")
        continue
    print(f"  {dataset_name:8s} provided {provided:6.2f}%   sklearn {library:6.2f}%   "
          f"difference {library - provided:+.2f} percentage points")

# Report

Cover the following for this notebook.

*   A table of the mean accuracy over the five folds, one row per dataset.
*   The scikit-learn mean accuracy beside the provided one, for the digits dataset and for Titanic.
*   Provided code against scikit-learn on each dataset: are the accuracies the same? If they differ, say what in the two implementations could explain it (variance smoothing, missing values, how unseen categories are handled).
*   The confusion matrix for each dataset.
*   The digit pairs that get confused most, read off the confusion matrix, and what makes them hard to tell apart.
*   Why a Gaussian fits the pixel data, and what the variance epsilon is doing there.
*   How you split the Titanic columns, and what each variant assumes about the columns you gave it.
*   How the two Titanic models combine into one score, and what goes wrong if the prior is counted twice.
*   What you would expect if every Titanic column went to the numeric variant instead.